# **Hepatic Vessel Segmentation on Contrast-Enhanced CT**

---
**Model:** nnU-Net v2 (Residual Encoder L, `3d_fullres`) fine-tuned on 25 annotated abdominal CT scans of living liver donors.

**Notebook author:** Xinzi He

---
<font color='red'>This code is strictly for research purposes and is NOT intended for clinical, diagnostic, or treatment use.</font>

# **Overview**
- This notebook segments four vascular structures on **contrast-enhanced abdominal CT**:

  | Label | Structure |
  |---|---|
  | 1 | Portal vein |
  | 2 | Inferior vena cava (IVC) |
  | 3 | Portal-splenic confluence |
  | 4 | Hepatic veins |

- The model is a **standard nnU-Net v2 model**. It runs with `pip install nnunetv2` and no custom code, so it also runs on your own GPU server (see step 8).
- For every scan it writes a label map `<scan>_vessels.nii.gz` on the same voxel grid as the input, a quality-control image `<scan>_qc.png`, and a volume table `hepatic_vessel_volumes.csv` (mL).
- Inputs may be **NIfTI** (`.nii.gz`, `.nii`), `.nrrd` / `.mha` files, or **DICOM** folders. Sub-folders are searched, and every DICOM series found is segmented separately.
- The model was trained on only 25 scans. Accuracy on other scanners, protocols or contrast phases has not been established, so **review every segmentation** (step 7, and ideally in ITK-SNAP or 3D Slicer).

The workflow: **(1)** enable a GPU runtime -> **(2)** install packages -> **(3)** download the model -> **(4)** point to your images on Google Drive -> **(5)** run segmentation -> **(6)** read the volume table -> **(7)** check the segmentations.

## **1. Ensure the GPU runtime is enabled.**
- If the GPU runtime of Colab is enabled, you'll see **'T4'** displayed in the upper right corner, just below the 'Share' button.
- If it's not enabled, follow these steps:
  1. Click on 'Runtime' > 'Change Runtime Type'.
  2. In the window that appears, choose either **'T4 GPU'** or **'A100 GPU'**.

## **2. Install the required packages.**
Run once per session (about 1-2 minutes). This installs the official nnU-Net release from PyPI.

In [ ]:
#@title Install packages (run once)
!pip install -q nnunetv2==2.8.1 ipyfilechooser

import os
WORK = "/content/hepatic_vessels"
for name in ("nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"):
    os.environ[name] = os.path.join(WORK, name)
    os.makedirs(os.environ[name], exist_ok=True)

import torch
if torch.cuda.is_available():
    print("Ready. GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU found. Select Runtime > Change runtime type > T4 GPU, then run this cell again.")

## **3. Download the model.**
The model (about 380 MB) is published as a **GitHub Release asset** of this repository and is installed with nnU-Net's own `nnUNetv2_install_pretrained_model_from_zip`. The download is checked against its SHA-256 checksum.

In [ ]:
#@title Download and install the model
import hashlib, os, subprocess

MODEL_URL = "https://github.com/WCM-HPB/Hepatic-Vessel-Segmentation/releases/download/hepatic-vessels-20260921/Dataset092_Hepatic_Vessels_25_nnUNetv2.zip"
MODEL_SHA256 = "6a9feb21305b9d4a2eca8c9dbe5923f84f754e71306d6558d826179b3ffbdb6f"
MODEL_DIR = os.path.join(os.environ["nnUNet_results"], "Dataset092_Hepatic_Vessels_25",
                         "nnUNetTrainer__nnUNetPlannerResEncL__3d_fullres")

if not os.path.exists(os.path.join(MODEL_DIR, "fold_all", "checkpoint_final.pth")):
    zip_path = os.path.join(WORK, "model.zip")
    subprocess.run(["curl", "-fsSL", "-o", zip_path, MODEL_URL], check=True)
    digest = hashlib.sha256()
    with open(zip_path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    assert digest.hexdigest() == MODEL_SHA256, "The model download is incomplete or corrupted - run this cell again."
    subprocess.run(["nnUNetv2_install_pretrained_model_from_zip", zip_path], check=True)
    os.remove(zip_path)
print("Model ready:", MODEL_DIR)

## **4. Point the notebook to your images on Google Drive.**
- Put your CT scans (NIfTI files, or DICOM folders) in a folder on your Google Drive.
- <font color='red'>For privacy, anonymize your images before uploading them.</font> ITK-SNAP can convert DICOM to NIfTI (File > Save Image > Main Image, format NIfTI), which drops patient identifiers.
- Run the cell below, then use the file browsers to select an **input** folder (your images) and an **output** folder. Results are written to a `hepatic_vessels` sub-folder of the output folder.

In [ ]:
#@title Select input and output folders
from google.colab import drive
from ipyfilechooser import FileChooser
from IPython.display import display

drive.mount('/content/drive')

fc_input = FileChooser('/content/drive')
fc_input.title = '<b>Select the INPUT folder (NIfTI files or DICOM folders)</b>'
display(fc_input)

fc_output = FileChooser('/content/drive/MyDrive')
fc_output.title = '<b>Select the OUTPUT folder (results go to its hepatic_vessels sub-folder)</b>'
fc_output.show_only_dirs = True
display(fc_output)

## **5. Run segmentation.**
For each scan the notebook reorients the image to RAS (the orientation the model was trained on), runs nnU-Net, and reorients the labels back so they line up voxel-for-voxel with your input. DICOM series are also saved as `<scan>_ct.nii.gz` so the labels can be overlaid on them.

Expect about 1-3 minutes per scan on a free T4 GPU with test-time mirroring switched on (thin-slice scans take longest). Scans with more than about 1,000 slices can exceed the 12.7 GB of RAM on the free tier; use a High-RAM runtime or crop them to the abdomen first.

In [ ]:
#@title Run segmentation
#@markdown **Test-time mirroring** averages 8 mirrored copies of each prediction, as in the model's validation. Untick it for a faster run.
test_time_mirroring = True  #@param {type:"boolean"}

import os, re, time
import numpy as np
import SimpleITK as sitk
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

assert fc_input.selected_path, "Please select an INPUT folder in step 4."
assert fc_output.selected_path, "Please select an OUTPUT folder in step 4."
input_folder = os.path.normpath(fc_input.selected_path)
result_folder = os.path.join(os.path.normpath(fc_output.selected_path), "hepatic_vessels")
os.makedirs(result_folder, exist_ok=True)

LABELS = {1: "Portal vein", 2: "IVC", 3: "Portal-splenic confluence", 4: "Hepatic veins"}
COLORS = {1: "#1f77b4", 2: "#2ca02c", 3: "#ff7f0e", 4: "#d62728"}
IMAGE_ENDINGS = (".nii.gz", ".nii", ".nrrd", ".mha", ".mhd")
MIN_SLICES = 20  # skips scouts and localizers


def safe_name(text):
    return re.sub(r"[^A-Za-z0-9.-]+", "_", text).strip("_") or "scan"


def find_scans(folder):
    # Every image file and every DICOM series under folder, as (name, source, is_dicom).
    scans = []
    for root, dirs, files in os.walk(folder):
        dirs[:] = sorted(d for d in dirs if os.path.join(root, d) != result_folder)
        rel = os.path.relpath(root, folder)
        where = "" if rel == "." else rel + "_"
        other_files = False
        for f in sorted(files):
            ending = next((e for e in IMAGE_ENDINGS if f.lower().endswith(e)), None)
            if ending:
                scans.append((where + f[: -len(ending)], os.path.join(root, f), False))
            elif not f.startswith("."):
                other_files = True
        for series in (sitk.ImageSeriesReader.GetGDCMSeriesIDs(root) if other_files else ()):
            dicom_files = list(sitk.ImageSeriesReader.GetGDCMSeriesFileNames(root, series))
            header = sitk.ImageFileReader()
            header.SetFileName(dicom_files[0])
            header.ReadImageInformation()
            size = header.GetSize()
            slices = len(dicom_files) if len(dicom_files) > 1 else (size[2] if len(size) > 2 else 1)
            if slices < MIN_SLICES:
                continue
            tag = lambda key: header.GetMetaData(key).strip() if header.HasMetaDataKey(key) else ""
            source = dicom_files if len(dicom_files) > 1 else dicom_files[0]  # one file = multi-frame DICOM
            series_name = f"S{tag('0020|0011')}_{tag('0008|103e')}"
            scans.append(((where or os.path.basename(folder) + "_") + series_name, source, True))
    unique, seen = [], set()
    for name, source, is_dicom in scans:
        base = candidate = safe_name(name)
        k = 2
        while candidate in seen:
            candidate, k = f"{base}_{k}", k + 1
        seen.add(candidate)
        unique.append((candidate, source, is_dicom))
    return unique


def read_scan(source):
    if isinstance(source, list):
        reader = sitk.ImageSeriesReader()
        reader.SetFileNames(source)
        image = reader.Execute()
    else:
        image = sitk.ReadImage(source)
    # nnU-Net computes in float32; float64 CTs would only double the memory footprint.
    return sitk.Cast(image, sitk.sitkFloat32) if image.GetPixelID() == sitk.sitkFloat64 else image


def segment(image):
    # The model was trained on arrays reoriented with SimpleITK.DICOMOrient(image, "RAS"): reorient,
    # predict with stock nnU-Net, then reorient the labels back onto the input's voxel grid.
    if image.GetDimension() != 3 or image.GetNumberOfComponentsPerPixel() != 1:
        raise ValueError("expected a 3D grayscale CT volume")
    orientation = sitk.DICOMOrientImageFilter.GetOrientationFromDirectionCosines(image.GetDirection())
    ras = sitk.DICOMOrient(image, "RAS")
    data = sitk.GetArrayFromImage(ras).astype(np.float32, copy=False)[None]
    geometry = ras.GetOrigin(), ras.GetSpacing(), ras.GetDirection()
    del ras  # free the copy before nnU-Net allocates its own buffers
    labels = predictor.predict_single_npy_array(data, {"spacing": list(geometry[1])[::-1]})
    seg = sitk.GetImageFromArray(labels.astype(np.uint8))
    seg.SetOrigin(geometry[0])
    seg.SetSpacing(geometry[1])
    seg.SetDirection(geometry[2])
    seg = sitk.DICOMOrient(seg, orientation)
    seg.CopyInformation(image)
    return seg


def save_qc(image, seg, name, path):
    # Axial and coronal slices through the portal-splenic confluence, plus a coronal projection of all labels.
    ct = sitk.DICOMOrient(image, "LPS")  # LPS arrays display anterior-up with the patient's right on the left
    arr = sitk.GetArrayViewFromImage(ct)
    lab = sitk.GetArrayFromImage(sitk.DICOMOrient(seg, "LPS"))
    sx, _, sz = ct.GetSpacing()
    focus = np.argwhere(lab == 3) if (lab == 3).any() else np.argwhere(lab > 0)
    z, y, _ = focus.mean(0).round().astype(int) if len(focus) else np.array(lab.shape) // 2
    projection = np.zeros((lab.shape[0], lab.shape[2]), np.uint8)
    for k in (2, 4, 1, 3):  # small structures last so they stay visible
        projection[(lab == k).any(axis=1)] = k

    cmap = ListedColormap([COLORS[k] for k in LABELS])
    norm = BoundaryNorm(np.arange(0.5, len(LABELS) + 1), cmap.N)
    panels = [  # title, CT, labels, pixel aspect, HU window max, label opacity
        ("Axial", arr[z], lab[z], 1.0, 300, 0.6),
        ("Coronal", arr[:, y][::-1], lab[:, y][::-1], sz / sx, 300, 0.6),
        ("Coronal projection", arr.max(axis=1)[::-1], projection[::-1], sz / sx, 1000, 0.8),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), layout="constrained")
    for ax, (title, img, lbl, aspect, vmax, alpha) in zip(axes, panels):
        ax.imshow(img, cmap="gray", vmin=-100, vmax=vmax, aspect=aspect)
        ax.imshow(np.ma.masked_equal(lbl, 0), cmap=cmap, norm=norm, alpha=alpha, aspect=aspect, interpolation="nearest")
        ax.set_title(title)
        ax.axis("off")
    handles = [plt.Rectangle((0, 0), 1, 1, color=COLORS[k]) for k in LABELS]
    fig.legend(handles, LABELS.values(), loc="outside lower center", ncol=len(LABELS), frameon=False)
    fig.suptitle(name)
    fig.savefig(path, dpi=80, bbox_inches="tight")
    plt.close(fig)


def process(name, source, is_dicom):
    # One scan per call, so its arrays are released before the next scan is read.
    image = read_scan(source)
    seg = segment(image)
    sitk.WriteImage(seg, os.path.join(result_folder, f"{name}_vessels.nii.gz"), True)
    if is_dicom:
        sitk.WriteImage(image, os.path.join(result_folder, f"{name}_ct.nii.gz"), True)
    save_qc(image, seg, name, os.path.join(result_folder, f"{name}_qc.png"))


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
predictor = nnUNetPredictor(tile_step_size=0.5, use_gaussian=True, use_mirroring=test_time_mirroring,
                            perform_everything_on_device=device.type == "cuda", device=device,
                            verbose=False, verbose_preprocessing=False, allow_tqdm=True)
predictor.initialize_from_trained_model_folder(MODEL_DIR, use_folds=("all",), checkpoint_name="checkpoint_final.pth")

scans = find_scans(input_folder)
print(f"Found {len(scans)} scan(s) in {input_folder}\n")
done = 0
for i, (name, source, is_dicom) in enumerate(scans, 1):
    start = time.time()
    try:
        process(name, source, is_dicom)
        done += 1
        print(f"[{i}/{len(scans)}] {name}: done in {time.time() - start:.0f} s")
    except Exception as exc:
        print(f"[{i}/{len(scans)}] {name}: FAILED - {exc}")
print(f"\n{done} of {len(scans)} scan(s) segmented. Results are in {result_folder}")

## **6. Vessel volume summary.**
Volumes are computed from every label map in the results folder (mL = voxel count x voxel volume / 1000) and saved as `hepatic_vessel_volumes.csv`.

In [ ]:
#@title Vessel volume table
import glob, os
import numpy as np
import pandas as pd
import SimpleITK as sitk
from IPython.display import display

LABELS = {1: "Portal vein", 2: "IVC", 3: "Portal-splenic confluence", 4: "Hepatic veins"}
rows = []
for path in sorted(glob.glob(os.path.join(result_folder, "*_vessels.nii.gz"))):
    seg = sitk.ReadImage(path)
    counts = np.bincount(sitk.GetArrayViewFromImage(seg).ravel(), minlength=len(LABELS) + 1)
    voxel_ml = float(np.prod(seg.GetSpacing())) / 1000.0
    row = {"Scan": os.path.basename(path)[: -len("_vessels.nii.gz")]}
    row.update({f"{name} (mL)": round(counts[k] * voxel_ml, 1) for k, name in LABELS.items()})
    rows.append(row)

df = pd.DataFrame(rows)
if df.empty:
    print("No segmentations found in", result_folder)
else:
    display(df)
    out_csv = os.path.join(result_folder, "hepatic_vessel_volumes.csv")
    df.to_csv(out_csv, index=False)
    print("Saved:", out_csv)

## **7. Check the segmentations.**
Each image shows an axial and a coronal slice through the portal-splenic confluence, and a coronal projection of all four structures (blue: portal vein, green: IVC, orange: portal-splenic confluence, red: hepatic veins). For a full review, open `<scan>_vessels.nii.gz` as a segmentation on top of the CT in ITK-SNAP or 3D Slicer.

In [ ]:
#@title Show quality-control images
import glob, os
from IPython.display import Image, display

pngs = sorted(glob.glob(os.path.join(result_folder, "*_qc.png")))
if not pngs:
    print("No quality-control images found in", result_folder)
for png in pngs:
    display(Image(filename=png))

## **8. Run on your own GPU server (optional).**
For privacy, you can run the same model on your own machine. It is a standard nnU-Net v2 model:

```bash
pip install nnunetv2==2.8.1
export nnUNet_results=/path/to/nnUNet_results
nnUNetv2_install_pretrained_model_from_zip Dataset092_Hepatic_Vessels_25_nnUNetv2.zip
```

The model was trained on images reoriented to **RAS** with `SimpleITK.DICOMOrient(image, "RAS")`, and `nnUNetv2_predict` does not reorient. Reorient your scans first (the `segment()` function in step 5 shows how), name them `<scan>_0000.nii.gz`, then run:

```bash
nnUNetv2_predict -i images_ras/ -o predictions/ -d 92 -c 3d_fullres -tr nnUNetTrainer -p nnUNetPlannerResEncL -f all
```

The predicted label maps carry the RAS geometry in their headers, so viewers such as ITK-SNAP and 3D Slicer overlay them correctly on the original scans.

# **Usage Guidelines**
This code is strictly for research purposes and is not intended for clinical, diagnostic, or treatment use. By using it, you agree not to use it for clinical, diagnostic, or treatment purposes. We accept no liability for the use of this code.

For questions or comments, please email <xh278@cornell.edu>.